In [1]:
pwd

'/mnt/stor/geob/jlmd9g/Rishabh/northslope/timeseries/dt102/SBAS'

In [10]:
# use slc-list to make new ifg pairs
# making pairs only 1 year apart ifg between May and October

In [5]:
import datetime

# Function to read dates from file
def read_dates(filename):
    with open(filename, 'r') as file:
        dates = [line.strip() for line in file.readlines()]
    return dates

# Function to check if the date is between May and October
def is_between_may_and_october(date):
    return 5 <= date.month <= 10

# Function to create pairs
def create_pairs(dates):
    pairs = []
    for i in range(len(dates)):
        current_date = datetime.datetime.strptime(dates[i], '%Y%m%d')
        if is_between_may_and_october(current_date):
            for j in range(i + 1, len(dates)):
                next_date = datetime.datetime.strptime(dates[j], '%Y%m%d')
                # Check if the next date is in the next year and the same, previous, or next month
                if next_date.year == current_date.year + 1 and (
                    next_date.month == current_date.month or
                    next_date.month == current_date.month - 1 or
                    next_date.month == current_date.month + 1
                ):
                    pairs.append(f"{dates[i]}_{dates[j]}")
    return pairs

# Function to write pairs to a new file
def write_pairs(filename, pairs):
    with open(filename, 'w') as file:
        for pair in pairs:
            file.write(pair + '\n')

# Main process
input_file = 'slc_list.txt'
output_file = 'pairs_list.txt'

dates = read_dates(input_file)
pairs = create_pairs(dates)
write_pairs(output_file, pairs)

print(f"Pairs have been written to {output_file}")


Pairs have been written to pairs_list.txt


In [11]:
# the number of ifgs are just too many
# divide them into multiple different files 

In [6]:
import os

# Function to read pairs from file
def read_pairs(filename):
    with open(filename, 'r') as file:
        pairs = [line.strip() for line in file.readlines()]
    return pairs

# Function to write pairs to a file
def write_pairs(filename, pairs):
    with open(filename, 'w') as file:
        for pair in pairs:
            file.write(pair + '\n')

# Main process
input_file = 'pairs_list.txt'
output_files = [f'pairs_list_part_{i+1}.txt' for i in range(5)]

pairs = read_pairs(input_file)
total_pairs = len(pairs)
pairs_per_file = total_pairs // 5
remainder = total_pairs % 5

start_idx = 0

for i in range(5):
    end_idx = start_idx + pairs_per_file + (1 if i < remainder else 0)
    write_pairs(output_files[i], pairs[start_idx:end_idx])
    start_idx = end_idx

print(f"Pairs have been divided into {len(output_files)} files.")


Pairs have been divided into 5 files.


In [12]:
# now create the lines to run make_ifg.py

!awk -F_ '{print "python make_ifg.py -m " $1 " -s " $2}' pairs_list.txt > make_ifg_commands.txt